In [32]:
# 1. Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib

# 2. Load reduced dataset
df = pd.read_csv("../data/reduced_heart_disease.csv")

X = df.drop("num", axis=1)
y = df["num"]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)


In [33]:
baseline_models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="liblinear"),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(probability=True, random_state=42)
}

baseline_results = {}

for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    
    baseline_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    }

print("Baseline Results:")
print(pd.DataFrame(baseline_results).T)


Baseline Results:
                     Accuracy  Precision    Recall  F1-Score       AUC
Logistic Regression  0.868852   0.812500  0.928571  0.866667  0.958874
Decision Tree        0.754098   0.709677  0.785714  0.745763  0.756494
Random Forest        0.885246   0.818182  0.964286  0.885246  0.936688
SVM                  0.901639   0.892857  0.892857  0.892857  0.958874


In [34]:
# Logistic Regression
log_reg_params = {"C":[0.01, 0.1, 1, 10], "penalty":["l1","l2"]}

# Decision Tree
dt_params = {"max_depth":[3,5,10,None], "min_samples_split":[2,5,10]}

# Random Forest
rf_params = {"n_estimators":[100,200,500],
             "max_depth":[5,10,None],
             "min_samples_split":[2,5,10]}

# SVM
svm_params = {"C":[0.1,1,10],
              "kernel":["linear","rbf"],
              "gamma":["scale","auto"]}


In [35]:
tuned_results = {}
best_models = {}

# Logistic Regression - GridSearch
grid_log = GridSearchCV(LogisticRegression(solver="liblinear", max_iter=2000), log_reg_params, cv=5, scoring="f1")
grid_log.fit(X_train, y_train)
best_models["Logistic Regression"] = grid_log.best_estimator_

# Decision Tree - GridSearch
grid_dt = GridSearchCV(DecisionTreeClassifier(random_state=42), dt_params, cv=5, scoring="f1")
grid_dt.fit(X_train, y_train)
best_models["Decision Tree"] = grid_dt.best_estimator_

# Random Forest - RandomizedSearch
rand_rf = RandomizedSearchCV(RandomForestClassifier(random_state=42), rf_params, n_iter=5, cv=5, scoring="f1", random_state=42)
rand_rf.fit(X_train, y_train)
best_models["Random Forest"] = rand_rf.best_estimator_

# SVM - RandomizedSearch
rand_svm = RandomizedSearchCV(SVC(probability=True, random_state=42), svm_params, n_iter=5, cv=5, scoring="f1", random_state=42)
rand_svm.fit(X_train, y_train)
best_models["SVM"] = rand_svm.best_estimator_


In [36]:
for name, model in best_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    
    tuned_results[name] = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred),
        "AUC": roc_auc_score(y_test, y_prob)
    }

print("Tuned Results:")
print(pd.DataFrame(tuned_results).T)


Tuned Results:
                     Accuracy  Precision    Recall  F1-Score       AUC
Logistic Regression  0.885246   0.838710  0.928571  0.881356  0.963203
Decision Tree        0.754098   0.696970  0.821429  0.754098  0.848485
Random Forest        0.868852   0.812500  0.928571  0.866667  0.943723
SVM                  0.852459   0.787879  0.928571  0.852459  0.956710


In [37]:
comparison_df = pd.DataFrame({
    "Baseline_Accuracy": {k:v["Accuracy"] for k,v in baseline_results.items()},
    "Tuned_Accuracy": {k:v["Accuracy"] for k,v in tuned_results.items()},
    "Baseline_F1": {k:v["F1-Score"] for k,v in baseline_results.items()},
    "Tuned_F1": {k:v["F1-Score"] for k,v in tuned_results.items()}
})

print(comparison_df)
comparison_df.to_csv("../results/hyperparameter_tuning_comparison.csv", index=True)


                     Baseline_Accuracy  Tuned_Accuracy  Baseline_F1  Tuned_F1
Logistic Regression           0.868852        0.885246     0.866667  0.881356
Decision Tree                 0.754098        0.754098     0.745763  0.754098
Random Forest                 0.885246        0.868852     0.885246  0.866667
SVM                           0.901639        0.852459     0.892857  0.852459


In [38]:
# Find best model by F1 score
best_model_name = max(tuned_results, key=lambda k: tuned_results[k]["F1-Score"])
best_model = best_models[best_model_name]

print(f"✅ Best model: {best_model_name} with F1 = {tuned_results[best_model_name]['F1-Score']:.3f}")

# Save best model
joblib.dump(best_model, f"../models/final_model.pkl")


✅ Best model: Logistic Regression with F1 = 0.881


['../models/final_model.pkl']

In [30]:
import joblib

# Save the best model (pipeline + classifier if possible)
joblib.dump(best_model, "../models/final_model.pkl")
print("Model saved successfully!")


Model saved successfully!


In [31]:
import os
os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/final_model.pkl")


['../models/final_model.pkl']